# MAF-OOD v5.3 multi-seed backbone sweep

この notebook は、提案手法 `MAF` を複数 backbone / 複数 seed で比較するための実験用です。

- 対象: `MAF` のみ
- alpha: `0.00..1.00` を `0.01` 刻み
- seed: `42, 123, 456`
- 既定 backbone: `imagenet_vit`, `openai_clip_b16`, `bioclip`, `dinov2_vitb14`

注意:
- `openai_clip_b16` と `dinov2_vitb14` は初回実行時に重みダウンロードが走ることがあります
- 現在の pipeline は `MAF_OOD_v51.ipynb` と同じ shared preprocessing / shared training recipe を使います
- したがってこれは backbone sweep 用の same-protocol 実験であり、各 backbone の native preprocessing 再現ではありません

今後追加候補:
- `siglip_base`
- `convnext_base_clip`


In [ ]:
# Dependency check
import importlib

required = [
    "numpy",
    "scipy",
    "sklearn",
    "torch",
    "torchvision",
    "timm",
    "open_clip",
    "pandas",
    "matplotlib",
    "seaborn",
]

missing = []
for name in required:
    try:
        importlib.import_module(name)
    except Exception as exc:
        missing.append((name, repr(exc)))

if missing:
    print("Missing or broken dependencies:")
    for name, exc in missing:
        print(f"- {name}: {exc}")
    raise RuntimeError("Fix the dependencies above before running the remaining cells.")

print("Dependency check passed.")


In [ ]:
# Config
from pathlib import Path
import numpy as np
import torch

SAVE_ROOT = str(Path("~/maf_ood_v51").expanduser())
DATA_SRC = str(Path("~/WILD_DATA/splits").expanduser())
BACKBONES = ["imagenet_vit", "openai_clip_b16", "bioclip", "dinov2_vitb14"]
SEEDS = [42, 123, 456]
ALPHA_GRID = np.round(np.arange(0.0, 1.0001, 0.01), 2)
FORCE_REEXTRACT = False
EVAL_ONLY = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CFG_DICT = {
    "epochs": 100,
    "batch_size": 64,
    "num_workers": 4,
    "lr": 0.005,
    "wd": 0.01,
    "lam": 0.5,
    "tau": 0.5,
}

print(f"SAVE_ROOT       : {SAVE_ROOT}")
print(f"DATA_SRC        : {DATA_SRC}")
print(f"BACKBONES       : {BACKBONES}")
print(f"SEEDS           : {SEEDS}")
print(f"ALPHA points    : {len(ALPHA_GRID)}")
print(f"FORCE_REEXTRACT : {FORCE_REEXTRACT}")
print(f"EVAL_ONLY       : {EVAL_ONLY}")
print(f"DEVICE          : {DEVICE}")

assert Path(DATA_SRC).exists(), f"not found: {DATA_SRC}"
Path(SAVE_ROOT).mkdir(parents=True, exist_ok=True)


In [ ]:
# Imports
import json
import pandas as pd
from IPython.display import display

import maf_ood_notebook_utils as nb
import maf_ood_dual_pipeline as core
from maf_ood_dual_pipeline import Cfg, ID_CLASSES, MAF, Mdl, load_bb, mkdl, train

print("Available backbones in registry:")
print(sorted(core.BACKBONES.keys()))


In [ ]:
# Helpers
def build_cfg(cfg_dict):
    return Cfg(
        lr=cfg_dict["lr"],
        wd=cfg_dict["wd"],
        epochs=cfg_dict["epochs"],
        bs=cfg_dict["batch_size"],
        nw=cfg_dict["num_workers"],
        lam=cfg_dict["lam"],
        tau=cfg_dict["tau"],
    )


def evaluate_maf_seed(
    backbone,
    seed,
    save_root,
    data_src,
    cfg,
    device,
    alpha_grid,
    force_reextract=False,
    eval_only=False,
):
    save_dir = Path(save_root).expanduser() / backbone / f"seed{seed}"
    save_dir.mkdir(parents=True, exist_ok=True)
    cache_path = save_dir / "analysis_v3.npz"

    loaders, _ = mkdl(data_src, cfg, seed)
    train_eval_loader = nb.build_train_eval_loader(data_src, cfg)
    bb, dim = load_bb(backbone, device)
    model = Mdl(bb, dim).to(device)

    best_path = save_dir / "best.pt"
    if best_path.exists():
        checkpoint = torch.load(best_path, map_location=device, weights_only=True)
        model.cls.load_state_dict(checkpoint["cls"])
        model.proj.load_state_dict(checkpoint["proj"])
    elif not eval_only:
        train(model, loaders, str(save_dir), seed, cfg, device)
        checkpoint = torch.load(best_path, map_location=device, weights_only=True)
        model.cls.load_state_dict(checkpoint["cls"])
        model.proj.load_state_dict(checkpoint["proj"])
    else:
        raise FileNotFoundError(f"Missing checkpoint: {best_path}")

    if force_reextract or not nb.cache_has_required_keys(cache_path):
        payload = {
            "tr": nb.extract_full(model, train_eval_loader, device),
            "val": nb.extract_full(model, loaders["val"], device),
            "id": nb.extract_full(model, loaders["test_id"], device),
            "ood": nb.extract_full(model, loaders["test_ood"], device),
        }
        nb._save_bundle_payload(payload, cache_path)
    payload = nb._load_bundle_payload(cache_path)

    readout = nb.fit_proj_readout(payload["tr"].proj, payload["tr"].logits)
    train_bundle = nb.attach_proj_logits(payload["tr"], readout)
    val_bundle = nb.attach_proj_logits(payload["val"], readout)
    id_bundle = nb.attach_proj_logits(payload["id"], readout)
    ood_bundle = nb.attach_proj_logits(payload["ood"], readout)

    proposal_raw_stats = nb.compute_space_stats(val_bundle.features, val_bundle.labels, train_bundle.features)
    maf = MAF(proposal_raw_stats.mu, proposal_raw_stats.covs, proposal_raw_stats.tied)

    alpha_rows = []
    best_alpha = None
    best_metrics = None
    for alpha in alpha_grid:
        metrics = nb.evaluate_scores(
            maf.score(id_bundle.features, "mah_t", 1.0, float(alpha)),
            maf.score(ood_bundle.features, "mah_t", 1.0, float(alpha)),
        )
        alpha_rows.append({"backbone": backbone, "seed": seed, "alpha": float(alpha), **metrics})
        if best_metrics is None or nb.score_sort_key(metrics) < nb.score_sort_key(best_metrics):
            best_alpha = float(alpha)
            best_metrics = metrics

    if best_alpha is None or best_metrics is None:
        raise RuntimeError("Failed to evaluate the MAF alpha sweep.")

    raw_acc_df, raw_cm = nb.classification_summary(id_bundle.labels, id_bundle.preds, ID_CLASSES)
    proj_acc_df, proj_cm = nb.classification_summary(id_bundle.labels, id_bundle.proj_preds, ID_CLASSES)

    raw_overall = float(raw_acc_df.loc[raw_acc_df["class"] == "overall", "accuracy"].iloc[0])
    proj_overall = float(proj_acc_df.loc[proj_acc_df["class"] == "overall", "accuracy"].iloc[0])

    out_dir = save_dir / "maf_multiseed_backbone_sweep_outputs"
    out_dir.mkdir(parents=True, exist_ok=True)
    alpha_df = pd.DataFrame(alpha_rows)

    alpha_csv = out_dir / f"maf_alpha_sweep_seed{seed}.csv"
    raw_acc_csv = out_dir / f"raw_id_accuracy_seed{seed}.csv"
    proj_acc_csv = out_dir / f"proj_id_accuracy_seed{seed}.csv"
    raw_cm_csv = out_dir / f"raw_id_confusion_seed{seed}.csv"
    proj_cm_csv = out_dir / f"proj_id_confusion_seed{seed}.csv"
    alpha_png = out_dir / f"maf_alpha_sweep_seed{seed}.png"
    raw_acc_png = out_dir / f"raw_id_accuracy_seed{seed}.png"
    proj_acc_png = out_dir / f"proj_id_accuracy_seed{seed}.png"
    raw_cm_png = out_dir / f"raw_id_confusion_seed{seed}.png"
    proj_cm_png = out_dir / f"proj_id_confusion_seed{seed}.png"
    summary_json = out_dir / f"summary_seed{seed}.json"

    alpha_df.to_csv(alpha_csv, index=False)
    raw_acc_df.to_csv(raw_acc_csv, index=False)
    proj_acc_df.to_csv(proj_acc_csv, index=False)
    pd.DataFrame(raw_cm, index=ID_CLASSES, columns=ID_CLASSES).to_csv(raw_cm_csv)
    pd.DataFrame(proj_cm, index=ID_CLASSES, columns=ID_CLASSES).to_csv(proj_cm_csv)

    nb.plot_maf_alpha_sweep(alpha_df, title=f"{backbone} / seed={seed} / MAF alpha sweep", out_path=str(alpha_png), show=False)
    nb.plot_class_accuracy(raw_acc_df, title=f"{backbone} / seed={seed} / raw-head class accuracy", out_path=str(raw_acc_png), show=False)
    nb.plot_class_accuracy(proj_acc_df, title=f"{backbone} / seed={seed} / projected-readout class accuracy", out_path=str(proj_acc_png), show=False)
    nb.plot_confusion(raw_cm, ID_CLASSES, title=f"{backbone} / seed={seed} / raw-head confusion", out_path=str(raw_cm_png), show=False)
    nb.plot_confusion(proj_cm, ID_CLASSES, title=f"{backbone} / seed={seed} / projected-readout confusion", out_path=str(proj_cm_png), show=False)

    best_row = {
        "backbone": backbone,
        "seed": seed,
        "method": "MAF Mah(tied)",
        "track": "Proposal",
        "best_alpha": best_alpha,
        "AUROC": float(best_metrics["AUROC"]),
        "AUPR-IN": float(best_metrics["AUPR-IN"]),
        "AUPR-OUT": float(best_metrics["AUPR-OUT"]),
        "FPR95": float(best_metrics["FPR95"]),
        "AUTC": float(best_metrics["AUTC"]),
        "raw_id_accuracy": raw_overall,
        "proj_id_accuracy": proj_overall,
        "cache_path": str(cache_path),
        "checkpoint": str(best_path),
        "note": f"best alpha from 0.00-1.00 step 0.01: {best_alpha:.2f}",
    }

    summary_json.write_text(
        json.dumps(
            {
                "backbone": backbone,
                "seed": seed,
                "best_row": best_row,
                "num_alpha_points": len(alpha_df),
                "artifacts": {
                    "alpha_csv": str(alpha_csv),
                    "raw_accuracy_csv": str(raw_acc_csv),
                    "proj_accuracy_csv": str(proj_acc_csv),
                    "raw_confusion_csv": str(raw_cm_csv),
                    "proj_confusion_csv": str(proj_cm_csv),
                    "alpha_png": str(alpha_png),
                    "raw_accuracy_png": str(raw_acc_png),
                    "proj_accuracy_png": str(proj_acc_png),
                    "raw_confusion_png": str(raw_cm_png),
                    "proj_confusion_png": str(proj_cm_png),
                },
            },
            indent=2,
            ensure_ascii=False,
        )
    )

    del bb, model
    torch.cuda.empty_cache()

    return {
        "backbone": backbone,
        "seed": seed,
        "best_row": best_row,
        "alpha_sweep": alpha_df,
        "raw_accuracy": raw_acc_df,
        "proj_accuracy": proj_acc_df,
        "raw_confusion": raw_cm,
        "proj_confusion": proj_cm,
        "artifacts": {
            "alpha_csv": str(alpha_csv),
            "raw_accuracy_csv": str(raw_acc_csv),
            "proj_accuracy_csv": str(proj_acc_csv),
            "raw_confusion_csv": str(raw_cm_csv),
            "proj_confusion_csv": str(proj_cm_csv),
            "alpha_png": str(alpha_png),
            "raw_accuracy_png": str(raw_acc_png),
            "proj_accuracy_png": str(proj_acc_png),
            "raw_confusion_png": str(raw_cm_png),
            "proj_confusion_png": str(proj_cm_png),
            "summary_json": str(summary_json),
        },
    }


In [ ]:
# Run the sweep
cfg = build_cfg(CFG_DICT)
runs = {}
best_rows = []
alpha_frames = []

for backbone in BACKBONES:
    for seed in SEEDS:
        print("=" * 120)
        print(f"Running backbone={backbone} seed={seed}")
        print("=" * 120)
        result = evaluate_maf_seed(
            backbone=backbone,
            seed=seed,
            save_root=SAVE_ROOT,
            data_src=DATA_SRC,
            cfg=cfg,
            device=DEVICE,
            alpha_grid=ALPHA_GRID,
            force_reextract=FORCE_REEXTRACT,
            eval_only=EVAL_ONLY,
        )
        runs[(backbone, seed)] = result
        best_rows.append(result["best_row"])
        alpha_frames.append(result["alpha_sweep"])

results_df = pd.DataFrame(best_rows).sort_values(
    by=["AUROC", "FPR95", "AUPR-OUT", "AUTC"],
    ascending=[False, True, False, True],
).reset_index(drop=True)
alpha_long_df = pd.concat(alpha_frames, ignore_index=True)

display(results_df)


In [ ]:
# Save and summarize
summary_root = Path(SAVE_ROOT).expanduser() / "maf_multiseed_backbone_sweep_summary"
summary_root.mkdir(parents=True, exist_ok=True)

results_csv = summary_root / "maf_best_rows_all.csv"
alpha_csv = summary_root / "maf_alpha_sweep_all.csv"
aggregate_csv = summary_root / "maf_backbone_summary_mean_std.csv"
run_meta_json = summary_root / "run_meta.json"

results_df.to_csv(results_csv, index=False)
alpha_long_df.to_csv(alpha_csv, index=False)

aggregate_df = results_df.groupby("backbone", dropna=False).agg(
    AUROC_mean=("AUROC", "mean"),
    AUROC_std=("AUROC", "std"),
    AUPRIN_mean=("AUPR-IN", "mean"),
    AUPRIN_std=("AUPR-IN", "std"),
    AUPROUT_mean=("AUPR-OUT", "mean"),
    AUPROUT_std=("AUPR-OUT", "std"),
    FPR95_mean=("FPR95", "mean"),
    FPR95_std=("FPR95", "std"),
    AUTC_mean=("AUTC", "mean"),
    AUTC_std=("AUTC", "std"),
    raw_id_accuracy_mean=("raw_id_accuracy", "mean"),
    raw_id_accuracy_std=("raw_id_accuracy", "std"),
    proj_id_accuracy_mean=("proj_id_accuracy", "mean"),
    proj_id_accuracy_std=("proj_id_accuracy", "std"),
    best_alpha_mean=("best_alpha", "mean"),
    best_alpha_std=("best_alpha", "std"),
).reset_index()

alpha_lists = (
    results_df.groupby("backbone")["best_alpha"]
    .apply(lambda s: ", ".join(f"{x:.2f}" for x in s.tolist()))
    .rename("best_alpha_values")
    .reset_index()
)
aggregate_df = aggregate_df.merge(alpha_lists, on="backbone", how="left")
aggregate_df = aggregate_df.sort_values(by=["AUROC_mean", "FPR95_mean"], ascending=[False, True]).reset_index(drop=True)
aggregate_df.to_csv(aggregate_csv, index=False)

run_meta_json.write_text(
    json.dumps(
        {
            "save_root": SAVE_ROOT,
            "data_src": DATA_SRC,
            "backbones": BACKBONES,
            "seeds": SEEDS,
            "alpha_grid": ALPHA_GRID.tolist(),
            "force_reextract": bool(FORCE_REEXTRACT),
            "eval_only": bool(EVAL_ONLY),
            "device": DEVICE,
            "cfg": CFG_DICT,
            "summary_files": {
                "results_csv": str(results_csv),
                "alpha_csv": str(alpha_csv),
                "aggregate_csv": str(aggregate_csv),
            },
        },
        indent=2,
        ensure_ascii=False,
    )
)

display(aggregate_df)
print(f"Saved: {results_csv}")
print(f"Saved: {alpha_csv}")
print(f"Saved: {aggregate_csv}")
print(f"Saved: {run_meta_json}")


In [ ]:
# Alpha curves by backbone / seed
import matplotlib.pyplot as plt
import seaborn as sns

alpha_plot_png = summary_root / "maf_alpha_curves_by_backbone_seed.png"
fig, axes = plt.subplots(len(BACKBONES), 1, figsize=(9, max(4, 3 * len(BACKBONES))), sharex=True, sharey=True)
if len(BACKBONES) == 1:
    axes = [axes]

for ax, backbone in zip(axes, BACKBONES):
    subset = alpha_long_df[alpha_long_df["backbone"] == backbone].copy()
    sns.lineplot(data=subset, x="alpha", y="AUROC", hue="seed", marker="o", linewidth=1.4, markersize=3, ax=ax)
    best_rows = results_df[results_df["backbone"] == backbone]
    for _, row in best_rows.iterrows():
        ax.axvline(row["best_alpha"], linestyle="--", linewidth=0.8, alpha=0.35)
    ax.set_title(backbone)
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("AUROC")

axes[-1].set_xlabel("alpha")
plt.tight_layout()
plt.savefig(alpha_plot_png, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {alpha_plot_png}")


In [ ]:
# Inspect one backbone / seed in detail
SHOW_BACKBONE = BACKBONES[0]
SHOW_SEED = SEEDS[0]
selected = runs[(SHOW_BACKBONE, SHOW_SEED)]

print(f"Selected: backbone={SHOW_BACKBONE}, seed={SHOW_SEED}")
display(pd.DataFrame([selected["best_row"]]))
display(selected["alpha_sweep"].sort_values(by=["AUROC", "FPR95", "AUPR-OUT", "AUTC"], ascending=[False, True, False, True]).head(10))
display(selected["raw_accuracy"])
display(selected["proj_accuracy"])

nb.plot_maf_alpha_sweep(selected["alpha_sweep"], title=f"{SHOW_BACKBONE} / seed={SHOW_SEED} / MAF alpha sweep")
nb.plot_class_accuracy(selected["raw_accuracy"], title=f"{SHOW_BACKBONE} / seed={SHOW_SEED} / raw-head class accuracy")
nb.plot_class_accuracy(selected["proj_accuracy"], title=f"{SHOW_BACKBONE} / seed={SHOW_SEED} / projected-readout class accuracy")
nb.plot_confusion(selected["raw_confusion"], ID_CLASSES, title=f"{SHOW_BACKBONE} / seed={SHOW_SEED} / raw-head confusion")
nb.plot_confusion(selected["proj_confusion"], ID_CLASSES, title=f"{SHOW_BACKBONE} / seed={SHOW_SEED} / projected-readout confusion")


## Recommended next ablations

この notebook の次に見る価値が高い ablation は次です。

1. `OpenAI CLIP ViT-B/16` vs `BioCLIP`
2. `ImageNet supervised` vs `DINOv2` vs `CLIP / BioCLIP`
3. shared preprocessing vs backbone-native preprocessing
4. `Proj dim = 64 / 128 / 256`
5. `MAF` の composition (`adaptive`, `max`, `1-Hn`, `product`)
6. `MAF` の distance (`euc`, `mah_t`, classwise Mahalanobis)
